# Zero Tic-Tac-Toe — NRLS Solver Demo
Train a shallow evaluator with **NRLS** to imitate an exact solver.

In [1]:
import sys, numpy as np
import zero_ttt_core as Z
import zero_ttt_nrls_utils as U
from zero_ttt_core import initial_state, legal_moves, apply_move, solve_exact, search_theta, FEATURE_NAMES
import nrls_optimizer_zero_ttt as NR

sys.path.append('/data')
np.set_printoptions(precision=3, suppress=True)

## Sample random state & exact best move

In [2]:
s = Z.random_reachable_state(seed=3, steps=5)
solve_exact(s)

(1, (3, 1))

## Train NRLS on 40 states (depth=4)

In [3]:
res, train_states = NR.train_nrls(seed=0, n_states=50, depth=4)
res.theta, res.value, res.evaluations
theta = res.theta

== Level 1/3 | grid=5, topk=6 ==
   best so far: -0.955 θ=[ 0.  -1.5  3.  -3.  -3.   0.   0.   0.   0.   0.   0.   0. ]
== Level 2/3 | grid=7, topk=6 ==
   best so far: -0.909 θ=[-0.4 -1.5  3.4 -3.4 -3.   0.4  0.   0.   0.   0.   0.   0. ]
== Level 3/3 | grid=9, topk=6 ==
   best so far: -0.909 θ=[-0.4 -1.5  3.4 -3.4 -3.   0.4  0.   0.   0.   0.   0.   0. ]


## Evaluate agreement on 20 held-out states

In [4]:
rng = np.random.default_rng(77)
test_states = [Z.random_reachable_state(seed=int(rng.integers(0,1e9)), steps=int(rng.integers(2,9))) for _ in range(20)]
ok=0; tot=0
for st in test_states:
    v1,m1 = solve_exact(st)
    if m1 is None: continue
    v2,m2 = search_theta(st, depth=4, theta=res.theta)
    ok += int(m2==m1); tot += 1
ok, tot, ok/max(1,tot)

(7, 15, 0.4666666666666667)

## Root suggestion from θ̂ (depth=4)

In [5]:
s0 = initial_state()
search_theta(s0, depth=4, theta=res.theta)

(2, (3, 1))

## Play Test

In [7]:
# Bot vs bot (X uses θ, O uses exact solver)
winner, traj, final_state = U.play_game(theta_X=theta,theta_O=theta, use_exact_O=True, depth_X=4, verbose=True)

Index map:
 0  1  2
 3  4  5
 6  7  8

Start:
 .  .  .
 .  .  .
 .  .  .

X plays (value=3, index=1):
 . X3  .
 .  .  .
 .  .  .

O plays (value=3, index=4):
 . X3  .
 . O3  .
 .  .  .

X plays (value=2, index=3):
 . X3  .
X2 O3  .
 .  .  .

O plays (value=2, index=2):
 . X3 O2
X2 O3  .
 .  .  .

X plays (value=3, index=2):
 . X3 X3
X2 O3  .
 .  .  .

O plays (value=2, index=0):
O2 X3 X3
X2 O3  .
 .  .  .

X plays (value=1, index=8):
O2 X3 X3
X2 O3  .
 .  . X1

O plays (value=3, index=8):
O2 X3 X3
X2 O3  .
 .  . O3

O wins!
Final:
O2 X3 X3
X2 O3  .
 .  . O3


In [8]:
# Play against the bot (bot is O)
U.human_vs_bot(theta, bot_player='O', depth=4)

Index map:
 0  1  2
 3  4  5
 6  7  8

Start:
 .  .  .
 .  .  .
 .  .  .

You played: (3, 4)
 .  .  .
 . X3  .
 .  .  .

Bot plays (2, 1):
 . O2  .
 . X3  .
 .  .  .

You played: (1, 6)
 . O2  .
 . X3  .
X1  .  .

Bot plays (3, 2):
 . O2 O3
 . X3  .
X1  .  .

You played: (3, 0)
X3 O2 O3
 . X3  .
X1  .  .

Bot plays (3, 8):
X3 O2 O3
 . X3  .
X1  . O3

You played: (2, 3)
X3 O2 O3
X2 X3  .
X1  . O3

You win!
Final:
X3 O2 O3
X2 X3  .
X1  . O3


1

## Save Training Result

In [6]:
import json, time, numpy as np

theta = res.theta

# binary (fast/precise) — use this for loading later
np.save("zero_ttt_theta_new_objective.npy", theta)

# human-readable metadata (nice for experiment tracking)
meta = {
    "feature_names": Z.FEATURE_NAMES,
    "theta": theta.tolist(),
    "agreement": float(res.value),
    "evaluations": int(res.evaluations),
    "levels": (5,7,9),
    "topk": 6,
    "shrink": 0.4,
    "depth": 4,
    "seed": 0,
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
}
with open("zero_ttt_nrls_result_new_objective.json", "w") as f:
    json.dump(meta, f, indent=2)
print("saved.")


saved.


## Load Training Result

In [7]:
import numpy as np, json
theta = np.load("zero_ttt_theta_depth4.npy")
# OR from JSON:
with open("zero_ttt_nrls_result.json") as f:
    meta = json.load(f)
theta = np.array(meta["theta"], dtype=float)

In [8]:
v, mv = Z.search_theta(Z.initial_state(), depth=4, theta=theta)